# Swing ZigZag — ML Tier 3 (OHLCV + Order Flow)

> For research only — not financial advice. Tier 3 adds order-flow features (signed trade volume, trade-size profile, CVD) on top of the Tier 1 OHLCV features. The hypothesis: OFI captures the flow that produces price moves, not just the price effect of it — which 15m candles can't see.

If features have signal anywhere on 15m crypto, OFI is where most of it lives. Empirically the gain over OHLCV-only is +5–15 percentage points of skill ratio in published research; if your run beats T1 by a few pp, that's a successful Tier 3.

Architecture vs Tier 1: identical except for one new step — build_orderflow_features downloads Bybit's bulk tick archive, aggregates to 15m OFI bars, and merges into the feature frame. Same labels, same CV, same GBM. The point is to isolate the contribution of order flow — train T1 and T3 on the same folds and compare OOS performance directly.

New features (13 columns):

- ofi_buy_volume_usd / ofi_sell_volume_usd / ofi_total_volume_usd
- ofi_imbalance = (buy − sell) / (buy + sell) ∈ [−1, +1]
- ofi_trade_count, ofi_avg_trade_usd, ofi_max_trade_usd
- ofi_large_trade_count (>$100K) — institutional flow proxy
- ofi_buy_trade_ratio — count-weighted imbalance (different from volume-weighted)
- ofi_imbalance_z50, ofi_volume_z50, ofi_count_z50 — 50-bar z-scores
- ofi_cvd_change_10 — change in cumulative volume delta over last 10 bars

Operational notes

Disk cost:
- 2 years of raw .gz files ≈ 30 GB under data/trades/ (gitignored).
- Per-day aggregates under data/ofi_aggregates/ are ~10 KB each (~7 MB total).
- Set keep_raw=False under the ## Build order-flow features section, the build_orderflow_features(...) after first run if disk is tight; the aggregates are all you need for re-runs.
- If you set keep_raw=False and later want to recompute OFI features with a different aggregator (e.g., change LARGE_TRADE_USD threshold, add a new column, switch bar size), you'll have to re-download. The cached aggregates are frozen with the columns they were built from.
- Safe pattern: run once with keep_raw=True, confirm your feature set is stable, then delete data/trades/ manually (or re-run with keep_raw=False) once you're sure.

Network cost:
- ~25 minutes to fetch 2 years on first run; pipeline is resumable, so a network hiccup just re-runs the missing days.

Live mode: 
- intentionally not wired
- Live T3 needs a Bybit WebSocket subscription to publicTrade.{SYMBOL} plus an in-memory aggregator — separate engineering work documented in the notebook's "Live execution" section. 
- The strategy raises a helpful error if you try to run T3 live without it.

## Table of Contents

1. [Configuration](#configuration)
2. [Load OHLCV](#load-ohlcv)
3. [Build order-flow features](#build-order-flow-features)
4. [Merge T3 feature frame](#merge-t3-feature-frame)
5. [Labels + CV (identical to T1)](#labels-cv)
6. [Train T1 and T3 GBMs on same folds](#train-models)
7. [OOS backtest comparison](#oos-backtest-comparison)
8. [Feature importance — OFI vs OHLCV](#feature-importance)
9. [Train final T3 model + save](#train-final)
10. [Live execution caveats](#live-execution)

## Configuration

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
REPO_ROOT = root
print(f"repo root: {REPO_ROOT}")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.providers.bybit import BybitFetcher
from engine.strategy_configurator import SwingMlParams
from engine.strategies import SwingMLStrategy

from engine.ml.features import (
    FEATURE_COLUMNS, FEATURE_COLUMNS_T3,
    build_feature_frame, build_feature_frame_t3,
)
from engine.ml.labels import (
    LABEL_HOLD, LABEL_LONG, LABEL_SHORT,
    calibrate_threshold, label_distribution, oracle_swing_labels,
)
from engine.ml.order_flow import (
    OFI_FEATURE_COLUMNS, build_orderflow_features, merge_orderflow_features,
)
from engine.ml.splits import PurgedKFold

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
SYMBOL   = "BTCUSDT"
INTERVAL = "15"
START    = "2024-05-19"   # bulk trade archive coverage is good from ~2020
END      = "2026-05-19"

DATA_DIR     = REPO_ROOT / "data"
MODEL_DIR    = REPO_ROOT / "ml_models"
TRADES_DIR   = DATA_DIR / "trades"
OFI_AGG_DIR  = DATA_DIR / "ofi_aggregates"
for d in (DATA_DIR, MODEL_DIR, TRADES_DIR, OFI_AGG_DIR):
    d.mkdir(exist_ok=True)

KLINE_CACHE = DATA_DIR / f"{SYMBOL}_{INTERVAL}m_{START}_{END}.pkl"
MODEL_T1    = MODEL_DIR / "swing_zz_ml.joblib"
MODEL_T3    = MODEL_DIR / "swing_zz_ml_t3.joblib"
print(f"kline cache    : {KLINE_CACHE}")
print(f"T3 model out   : {MODEL_T3}")

## Load OHLCV

Same source as the T1 notebook — 2 years of 15m klines. Cached on disk so this cell is a no-op on re-run.

In [ ]:
if KLINE_CACHE.exists():
    df = pd.read_pickle(KLINE_CACHE)
    print(f"loaded cached {len(df):,} bars")
else:
    print(f"fetching klines from Bybit (1–3 min) ...")
    fetcher = BybitFetcher()
    df = fetcher.fetch_klines(
        symbol=SYMBOL, interval=INTERVAL, start_time=START, end_time=END,
    )
    fetcher.close()
    df.to_pickle(KLINE_CACHE)
    print(f"fetched {len(df):,} bars, cached")

print(f"range: {df.index[0]} → {df.index[-1]}")

## Build order-flow features

Download Bybit's bulk tick archive day-by-day, aggregate to 15m OFI bars, cache per-day pickle files. First run takes ~25 min for 2 years (depends on network); subsequent runs are instant from the per-day cache.

Raw daily .gz files (~30–80 MB each) live under data/trades/ — gitignored, ~30 GB total for 2 years. Delete that directory after this cell finishes if disk space is tight; the per-day aggregates (data/ofi_aggregates/, ~10 KB each) are all that's needed for re-runs.

In [ ]:
ofi = build_orderflow_features(
    symbol=SYMBOL,
    start=START,
    end=END,
    interval="15min",
    raw_dir=TRADES_DIR,
    agg_dir=OFI_AGG_DIR,
    keep_raw=True,     # set False to delete raw .gz after aggregating (saves disk)
    progress=True,
)
print(f"\nOFI shape: {ofi.shape}")
print(f"date range: {ofi.index[0]} → {ofi.index[-1]}")
ofi.tail(3)

In [ ]:
# Quick distribution sanity check.
print("imbalance stats:")
print(ofi['ofi_imbalance'].describe().round(3).to_string())
print("\nvolume stats (USD):")
print(ofi['ofi_total_volume_usd'].describe().round(0).to_string())
print("\nlarge-trade count stats:")
print(ofi['ofi_large_trade_count'].describe().round(1).to_string())

## Merge T3 feature frame

T3 = T1 OHLCV features ⊕ OFI features. The merge fills empty bars (no trades) with neutral zeros — "no flow" is a real state, not a missing-data error. After warmup, every column should be finite.

In [ ]:
features_t1 = build_feature_frame(df)
features_t3 = build_feature_frame_t3(df, ofi)

valid_mask = features_t3.notna().all(axis=1)
X_t1 = features_t1.loc[valid_mask]
X_t3 = features_t3.loc[valid_mask]
ts = df.index[valid_mask]
print(f"T1 features: {X_t1.shape}, T3 features: {X_t3.shape}")
print(f"OFI columns added: {len(FEATURE_COLUMNS_T3) - len(FEATURE_COLUMNS)}")
X_t3.tail(3)

## Labels + CV

Use the exact same oracle labels and the exact same purged k-fold splits the T1 notebook used. This is critical — we want to isolate the effect of adding OFI features, not the effect of label/CV differences.

In [ ]:
TARGET_PIVOTS = int(len(df) / 96 * 3)   # ~3 trades/day
ORACLE_MIN_PROM = calibrate_threshold(df, target_pivots=TARGET_PIVOTS)
labels_full = oracle_swing_labels(df, min_prominence_atr=ORACLE_MIN_PROM)
y = labels_full.loc[valid_mask]
print(f"min_prominence_atr: {ORACLE_MIN_PROM:.3f}")
print(f"label distribution: {label_distribution(y)}")

# purge=200 = the longest feature lookback (vol_200): drop train bars whose backward
# feature window overlaps the test fold; embargo=24 adds de Prado's after-fold buffer
# (audit M2 — embargo=100 alone was < 200, so vol_200 / atr_z / ret_100 still leaked).
splitter = PurgedKFold(n_splits=5, embargo=24, purge=200)
folds = list(splitter.split(len(X_t3)))
print(f"folds: {[(len(tr), len(te)) for tr, te in folds]}")

## Train models

Identical GBM hyperparameters, identical CV folds, identical sample weights — only the feature matrix differs. The OOS metric difference between T1 and T3 is purely the effect of adding order flow.

In [ ]:
def make_sample_weights(y_train):
    classes = np.array([LABEL_SHORT, LABEL_HOLD, LABEL_LONG])
    weights = compute_class_weight(
        class_weight="balanced", classes=classes, y=y_train.to_numpy(),
    )
    return y_train.map(dict(zip(classes, weights))).to_numpy()

def make_gbm():
    return HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=200, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=20, random_state=RNG_SEED,
    )

def evaluate(X, y, folds, name):
    proba_oos = np.full((len(X), 3), np.nan)
    classes_global = np.array([LABEL_SHORT, LABEL_HOLD, LABEL_LONG])
    accs = []
    for k, (tr, te) in enumerate(folds):
        m = make_gbm()
        sw = make_sample_weights(y.iloc[tr])
        m.fit(X.iloc[tr], y.iloc[tr], sample_weight=sw)
        proba = m.predict_proba(X.iloc[te])
        for j, cls in enumerate(classes_global):
            if cls in m.classes_:
                proba_oos[te, j] = proba[:, list(m.classes_).index(cls)]
            else:
                proba_oos[te, j] = 0.0
        acc = float((m.predict(X.iloc[te]) == y.iloc[te].to_numpy()).mean())
        accs.append(acc)
        print(f"  {name} fold {k}: acc={acc:.3f}, n_iter={m.n_iter_}")
    print(f"  {name} mean accuracy: {np.mean(accs):.3f}")
    return proba_oos

print("=== Tier 1 GBM (OHLCV only) ===")
t1_proba = evaluate(X_t1, y, folds, "T1")
print("\n=== Tier 3 GBM (OHLCV + OFI) ===")
t3_proba = evaluate(X_t3, y, folds, "T3")

In [ ]:
# Per-class OOS report, T3.
valid_oos = ~np.isnan(t3_proba).any(axis=1)
classes_global = np.array([LABEL_SHORT, LABEL_HOLD, LABEL_LONG])
y_pred_t3 = classes_global[t3_proba[valid_oos].argmax(axis=1)]
y_pred_t1 = classes_global[t1_proba[valid_oos].argmax(axis=1)]
y_oos = y.iloc[valid_oos].to_numpy()

print("\nT3 (OOS) classification report:")
print(classification_report(
    y_oos, y_pred_t3,
    labels=[LABEL_SHORT, LABEL_HOLD, LABEL_LONG],
    target_names=["short", "hold", "long"],
    digits=3, zero_division=0,
))
print("T1 (OOS) classification report:")
print(classification_report(
    y_oos, y_pred_t1,
    labels=[LABEL_SHORT, LABEL_HOLD, LABEL_LONG],
    target_names=["short", "hold", "long"],
    digits=3, zero_division=0,
))

## OOS backtest comparison

Run the OOS probabilities through the real SwingMLStrategy via the same Backtester everything else uses (probabilities are injected, so require_model=False skips model inference). The total-P&L gap between T1 and T3 is what order flow actually bought you.

Caveat: these numbers inherit direction and the risk overlays from trade_configurator.ACTIVE_TRADE — they match a plain both-sides run only while it keeps direction=both with max_holding_bars / max_daily_loss_bps off.

In [ ]:
def backtest_proba(df_full, ts_clean, proba_oos, threshold, use_stop=True):
    """Backtest OOS probabilities through the REAL strategy + Backtester (no
    duplicated loop). Trade-level policy — direction gate, max_holding / daily-loss
    overlays, costs — comes from ACTIVE_TRADE; editing it (e.g. direction=long or
    max_holding_bars) changes these OOS results too. Returns result.trades.
    """
    p_long_full = np.zeros(len(df_full))
    p_short_full = np.zeros(len(df_full))
    valid_full = np.zeros(len(df_full), dtype=np.int8)
    valid_oos = ~np.isnan(proba_oos).any(axis=1)
    iloc = df_full.index.get_indexer(ts_clean[valid_oos])
    p_long_full[iloc] = proba_oos[valid_oos, 2]
    p_short_full[iloc] = proba_oos[valid_oos, 0]
    valid_full[iloc] = 1

    prepared = df_full.copy()
    prepared["ml_p_long"] = p_long_full
    prepared["ml_p_short"] = p_short_full
    prepared["ml_valid"] = valid_full

    # Real strategy + real Backtester on the injected OOS probabilities — same
    # on_bar / exit_policy / cost as the live strategy (no duplicated loop to
    # drift). require_model=False: probabilities are supplied, not inferred.
    cfg = SwingMlParams(ml_p_threshold=threshold, ml_use_stop=use_stop)
    strat = SwingMLStrategy(cfg, require_model=False)
    result = Backtester(strat, symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(
        prepared, interval=INTERVAL,
    )
    return result.trades

def summary(trades, name):
    if not trades:
        return {"model": name, "trades": 0}
    pnls = np.array([t.pnl_bps for t in trades])
    wins = pnls[pnls > 0]; losses = pnls[pnls <= 0]
    pf = wins.sum() / abs(losses.sum()) if len(losses) and losses.sum() != 0 else float("inf")
    bal = 100.0; peak = bal; max_dd = 0.0
    for t in trades:
        bal *= (1 + t.pnl_bps / 10_000)
        peak = max(peak, bal)
        max_dd = max(max_dd, (peak - bal) / peak)
    return {
        "model": name, "trades": len(trades),
        "win_rate": round(len(wins) / len(trades) * 100, 1),
        "total_pnl_bps": round(pnls.sum(), 1),
        "avg_pnl_bps": round(pnls.mean(), 2),
        "profit_factor": round(pf, 2),
        "max_dd_pct": round(max_dd * 100, 2),
        "final_balance": round(bal, 2),
        "return_pct": round((bal / 100 - 1) * 100, 2),
    }

THRESHOLD = 0.55  # tighter than T1's 0.40 — most gain in T1 came from reducing trade count

t1_trades = backtest_proba(df, ts, t1_proba, threshold=THRESHOLD)
t3_trades = backtest_proba(df, ts, t3_proba, threshold=THRESHOLD)

compare = pd.DataFrame([
    summary(t1_trades, "T1 (OHLCV)"),
    summary(t3_trades, "T3 (OHLCV + OFI)"),
])
compare

In [ ]:
# Threshold sweep on T3 to find its operating point.
rows = []
for thr in (0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75):
    trades = backtest_proba(df, ts, t3_proba, threshold=thr)
    rows.append({**summary(trades, f"thr={thr}"), "threshold": thr})
thr_sweep = pd.DataFrame(rows).set_index("threshold")
thr_sweep

## Feature importance

Train one calibrated GBM on the full T3 data and run permutation importance. The question we care about: do OFI features rank in the top 10? If yes, order flow is contributing. If no, OFI is noise on this data / timeframe / regime.

In [ ]:
sw_full = make_sample_weights(y)
final_gbm = make_gbm()
final_gbm.fit(X_t3, y, sample_weight=sw_full)

# Permutation on a held-out tail.
split = int(len(X_t3) * 0.85)
perm = permutation_importance(
    final_gbm, X_t3.iloc[split:], y.iloc[split:],
    n_repeats=3, random_state=RNG_SEED, n_jobs=-1,
)
imp = pd.DataFrame({
    "feature": FEATURE_COLUMNS_T3,
    "perm_importance": perm.importances_mean,
    "family": ["OFI" if f in OFI_FEATURE_COLUMNS else "OHLCV" for f in FEATURE_COLUMNS_T3],
}).sort_values("perm_importance", ascending=False).head(20)
imp

In [ ]:
# Aggregate importance by family.
family_imp = pd.DataFrame({
    "feature": FEATURE_COLUMNS_T3,
    "perm_importance": perm.importances_mean,
    "family": ["OFI" if f in OFI_FEATURE_COLUMNS else "OHLCV" for f in FEATURE_COLUMNS_T3],
}).groupby("family").agg(total=("perm_importance", "sum"),
                          mean=("perm_importance", "mean"),
                          count=("perm_importance", "count"))
family_imp

## Train final

Calibrated GBM trained on the entire data window. The OOS evaluation above was on disjoint folds; the final production model gets to see all of it.

In [ ]:
final_model = CalibratedClassifierCV(
    make_gbm(), method="isotonic", cv=3,
)
final_model.fit(X_t3, y, sample_weight=sw_full)

bundle = {
    "model": final_model,
    "classes": final_model.classes_.tolist(),
    "features": tuple(FEATURE_COLUMNS_T3),
    "trained_on": {
        "symbol": SYMBOL, "interval": INTERVAL,
        "start": str(df.index[0]), "end": str(df.index[-1]),
        "n_bars": int(len(df)),
        "oracle_min_prominence_atr": float(ORACLE_MIN_PROM),
        "feature_schema": "t3_ohlcv_plus_orderflow",
    },
}
joblib.dump(bundle, MODEL_T3)
print(f"saved → {MODEL_T3}")

## Live execution

Backtesting with T3 already works — pre-merge OFI features into the OHLCV df, then pass to the existing SwingMLStrategy. The strategy auto-detects the schema from the model bundle and routes correctly.

Live mode for T3 is not yet wired. It needs:

1. A Bybit WebSocket subscription to publicTrade.{SYMBOL} so trades arrive in real time.
2. A rolling in-memory buffer that aggregates incoming trades into the same 15m bar grid.
3. Each LiveEngine._tick() would merge the latest aggregate row into the kline df before    calling strategy.prepare().

Until that's built, live mode with this model will raise the order-flow columns missing error. That's intentional — silent fallback to T1 features would be a much worse bug.

Backtest round-trip (this works today):

In [ ]:
# Pre-merge OFI features into the kline df, then run via the existing strategy.
df_with_ofi = pd.concat([
    df,
    merge_orderflow_features(df, ofi)[list(OFI_FEATURE_COLUMNS)],
], axis=1)

cfg = SwingMlParams(
    ml_model_path=str(MODEL_T3.relative_to(REPO_ROOT)),
    ml_p_threshold=THRESHOLD,
)
live_strategy = SwingMLStrategy(cfg)
result = Backtester(live_strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(
    df_with_ofi.tail(5000), interval=INTERVAL,
)
print(result.summary())

Decision criteria (what to look at before doing anything with this model):

1. T3 vs T1 total P&L gap. If T3 ≥ T1 by ≥ 1,000 bps over 2 years, OFI is helping.
2. OFI features in top-10 permutation importance. If yes, the model is actually using them.
3. Threshold sweep monotonic. Higher threshold → fewer trades → higher per-trade edge. If it doesn't, the model has no calibrated confidence.
4. OOS Sharpe on the most recent 6 months. Aggregate fold metrics can hide recent regime drift; look at the tail specifically.

If none of these hold, the answer is: OHLCV + 15m-aggregated OFI is the ceiling of what this strategy class can extract on this data. Next step up the ladder is tick-level features (microprice, BBO depth changes, queue position) — not more OHLCV feature engineering.